# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using `output/cohort.csv` produced by `01_build_cohort`.


Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
To reduce multicollinearity and improve balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.

In [190]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


exp_name = "E8b_vascular_signal"
H = 5  # horizon years used in build_cohort notebook

OUT_DIR = os.path.join("..", "output")

cohort_file = f"cohort_{exp_name}_H{H}.csv"
DATA_PATH = os.path.join(OUT_DIR, cohort_file)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find cohort file: {DATA_PATH}\n"
        f"Make sure 01_build_cohort exported: {cohort_file}"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", cohort_file, df.shape)
print(df["label"].value_counts(dropna=False))
df.head()

Loaded: cohort_E8b_vascular_signal_H5.csv (38341, 74)
label
Censored         23300
NoDementia_Hy    11083
AD                3013
MixedOrOther       504
PD_LBD             289
Vascular           152
Name: count, dtype: int64


,NACCID,baseline_date,label,max_follow_years,has_H_followup,has_dementia_H,NACCAGE,SEX,RACE,HISPANIC,...,FOCLSIGN,HACHIN,HACHIN_GE_4,HACHIN_GE_7,STROKE_TIA_ANY,VASC_MED_COUNT,GAIT_ANY,NACCMOM,NACCDAD,SIVDFIND
0,NACC000011,2006-04-17,Censored,3.296372,False,False,62.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,0.0,0,NaN,NaN,NaN
1,NACC000034,2015-07-16,NoDementia_Hy,6.986995,True,False,79.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,0,2.0,0,1.0,1.0,0.0
2,NACC000067,2012-05-23,Censored,2.088980,False,False,60.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,1.0,1,1.0,0.0,NaN
3,NACC000073,2016-05-20,Censored,0.000000,False,False,44.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,0,0.0,0,0.0,0.0,0.0
4,NACC000138,2023-07-13,Censored,1.043121,False,False,54.0,2.0,50.0,1.0,...,NaN,NaN,NaN,NaN,0,0.0,0,0.0,0.0,0.0


In [191]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figs_baseline")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

## Keep only the 4-class task

In [192]:
def fix_categorical_codes(X):
    X = X.copy()
    CAT_CODE_VARS = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD"] if c in X.columns]
    STD_MISS = {-4, 9}

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        s = s.mask(s.isin(list(STD_MISS)), np.nan)
        X[c] = s

    if "RACE" in X.columns:
        X["RACE"] = X["RACE"].mask(X["RACE"].isin([99]), np.nan)

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        X[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

    if "EDUC" in X.columns:
        educ_num = pd.to_numeric(X["EDUC"], errors="coerce")
        #X["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

    X = X.replace({pd.NA: np.nan})
    for c in X.columns:
        if X[c].dtype == "bool":
            X[c] = X[c].astype(int)
    return X

In [193]:
import numpy as np
import pandas as pd


df4 = df.copy()  

# Keeping only non-censored outcome classes for training
KEEP = {"AD", "NoDementia_Hy", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Dropping IDs/labels
#y = df4["label"].astype(str)
# drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
#drop_cols = [c for c in [ "NACCID", "baseline_date", "label", "has_H_followup", "has_dementia_H", "max_follow_years", "abs_days_from_baseline"] if c in df4.columns]


def prepare_experiment_df(df, exp_name):
    df = df.copy()

    # columns that are always dropped
    drop_cols = [c for c in ["NACCID","baseline_date","label","has_H_followup","has_dementia_H","max_follow_years","abs_days_from_baseline", "NACCUDSD"]
                 if c in df.columns]

    # E8: engineered vascular burden; dropping raw components to avoid collinearity
    if exp_name == "E8_clean_vascular":
        raw_vasc = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
        drop_cols += [c for c in raw_vasc if c in df.columns]

        raw_meds = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
        drop_cols += [c for c in raw_meds if c in df.columns]

        # drop_cols += [c for c in ["GAITDIS"] if c in df.columns]

    # E9: exec ratio; Dropping raw Trails
    if exp_name == "E9_clean_exec_ratio":
        drop_cols += [c for c in ["TRAILA","TRAILB","TRAILA_missing","TRAILB_missing"] if c in df.columns]

    # E10: building exec-memory axis (train-only z-scoring later), then Dropping raw
    # Here, what is dropped is only marked AFTER computing engineered features per split.

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["label"].astype(str)

    return X, y


from sklearn.model_selection import train_test_split

X, y = prepare_experiment_df(df4, exp_name)

# Applying categorical cleanup before splitting
X = fix_categorical_codes(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train label counts:\n", y_train.value_counts())
print("Test label counts:\n", y_test.value_counts())



def z_from_train(train_s, s):
    mu = train_s.mean()
    sd = train_s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return s*0.0
    return (s - mu) / sd

if exp_name == "E10_exec_mem_axis_raw":
    needed = ["NACCMMSE", "NACCMOCA", "TRAILB"]
    missing = [c for c in needed if c not in X_train.columns]
    if missing:
        raise ValueError(f"E10 missing required columns: {missing}")

    # Z-scores using train stats only
    X_train["Z_TRAILB"] = z_from_train(X_train["TRAILB"], X_train["TRAILB"])
    X_test["Z_TRAILB"]  = z_from_train(X_train["TRAILB"], X_test["TRAILB"])

    mem_cols = [c for c in ["NACCMMSE","NACCMOCA"] if c in X_train.columns]
    Zm_train = []
    Zm_test  = []
    for c in mem_cols:
        Zm_train.append(z_from_train(X_train[c], X_train[c]))
        Zm_test.append(z_from_train(X_train[c], X_test[c]))

    X_train["Z_MEM"] = np.nanmean(np.vstack([z.values for z in Zm_train]), axis=0)
    X_test["Z_MEM"]  = np.nanmean(np.vstack([z.values for z in Zm_test]), axis=0)

    # Executive minus memory axis
    X_train["EXEC_MINUS_MEM"] = X_train["Z_TRAILB"] - X_train["Z_MEM"]
    X_test["EXEC_MINUS_MEM"]  = X_test["Z_TRAILB"]  - X_test["Z_MEM"]

    # Dropping raw components to avoid collinearity
    drop_raw = [
        "TRAILA","TRAILB","NACCMMSE","NACCMOCA",
        "TRAILA_missing","TRAILB_missing","NACCMMSE_missing","NACCMOCA_missing",
        "Z_TRAILB","Z_MEM"
    ]
    X_train = X_train.drop(columns=[c for c in drop_raw if c in X_train.columns])
    X_test  = X_test.drop(columns=[c for c in drop_raw if c in X_test.columns])

    print("E10 engineered EXEC_MINUS_MEM added. Shapes now:", X_train.shape, X_test.shape)

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

4-class rows: 14537
label
NoDementia_Hy    11083
AD                3013
PD_LBD             289
Vascular           152
Name: count, dtype: int64
Train: (11629, 67) Test: (2908, 67)
Train label counts:
 label
NoDementia_Hy    8866
AD               2410
PD_LBD            231
Vascular          122
Name: count, dtype: int64
Test label counts:
 label
NoDementia_Hy    2217
AD                603
PD_LBD             58
Vascular           30
Name: count, dtype: int64


In [194]:
csf_cols = ["CSFABETA", "CSFPTAU", "CSFTTAU"]
present_csf_cols = [c for c in csf_cols if c in df.columns]

if len(present_csf_cols) == len(csf_cols):
    df_csf = df[df[present_csf_cols].notna().all(axis=1)].copy()
    print("CSF subset size:", len(df_csf))
    print(df_csf["label"].value_counts())
else:
    print("CSF columns not present in this dataset:", 
          [c for c in csf_cols if c not in df.columns])

CSF columns not present in this dataset: ['CSFABETA', 'CSFPTAU', 'CSFTTAU']


## Split + preprocess + train

Train/test split

Preprocessing pipelines (sparse for LR/boosting)

In [195]:
# cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("string")]
# num_cols = [c for c in X_train.columns if c not in cat_cols]

import pandas as pd

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))
print("Num cols:", num_cols, "Cat cols:", cat_cols)

# print(sorted([c for c in X_train.columns if "VASC" in c or "EXEC" in c or "TRAIL" in c or "MMSE" in c or "MOCA" in c]))
# print(sorted([c for c in X_train.columns]))



Num cols: 64 Cat cols: 3
Num cols: ['NACCAGE', 'EDUC', 'NACCMMSE', 'NACCMOCA', 'TRAILA', 'TRAILB', 'ANIMALS', 'BOSTON', 'CDRSUM', 'CDRGLOB', 'NACCMMSE_missing', 'NACCMOCA_missing', 'TRAILA_missing', 'TRAILB_missing', 'CDRSUM_missing', 'COGFLUC', 'BEVHALL', 'BEREM', 'PARKSIGN', 'BRADY', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SLOWINGL', 'SLOWINGR', 'PARKGAIT', 'GAITDIS', 'DIABETES', 'HYPERTEN', 'HYPERCHO', 'CVAFIB', 'CBSTROKE', 'CBTIA', 'CVCHF', 'CVANGINA', 'CVBYPASS', 'CVANGIO', 'CVHVALVE', 'CVOTHR', 'APNEA', 'DEP2YRS', 'DIABET', 'POSTINST', 'NACCGDS', 'DIGBACCT', 'DIGBACLS', 'CVD', 'VASC', 'CVDCOG', 'CVDIMAG', 'IMAGMWMH', 'IMAGEWMH', 'FOCLSYM', 'FOCLSIGN', 'HACHIN', 'HACHIN_GE_4', 'HACHIN_GE_7', 'STROKE_TIA_ANY', 'VASC_MED_COUNT', 'GAIT_ANY', 'NACCMOM', 'NACCDAD', 'SIVDFIND'] Cat cols: ['SEX', 'RACE', 'HISPANIC']


Model A: Logistic Regression (final settings)

# Section 3A: Logistic Regression (custom weights, C=0.1)

In [196]:
from sklearn.linear_model import LogisticRegression

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_train, y_train)
pred_lr = clf_lr.predict(X_test)

print("=== Logistic Regression (custom weights, C=0.1) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))


=== Logistic Regression (custom weights, C=0.1) ===
               precision    recall  f1-score   support

           AD     0.6542    0.7247    0.6876       603
NoDementia_Hy     0.9314    0.8935    0.9121      2217
       PD_LBD     0.4691    0.6552    0.5468        58
     Vascular     0.2812    0.3000    0.2903        30

     accuracy                         0.8477      2908
    macro avg     0.5840    0.6434    0.6092      2908
 weighted avg     0.8580    0.8477    0.8518      2908

Balanced accuracy: 0.6433580100833458
[[ 437  133   21   12]
 [ 205 1981   21   10]
 [  13    6   38    1]
 [  13    7    1    9]]


# Section 3G: Tuned Logistic Regression

In [197]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

# 1) Created train/validation split from TRAIN only
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# 2) Fitted LR on inner-train

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_tr, y_tr)

# 3) Tuned vascular threshold on validation set
proba_val = clf_lr.predict_proba(X_val)
class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

thresholds = np.linspace(0.05, 0.50, 20)
results = []

for T in thresholds:
    preds = []
    for p in proba_val:
        if p[vas_idx] >= T:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    ba = balanced_accuracy_score(y_val, preds)

    # vascular recall
    yv = np.array(y_val)
    pv = np.array(preds)
    denom = np.sum(yv == "Vascular")
    vrec = (np.sum((yv == "Vascular") & (pv == "Vascular")) / denom) if denom else 0.0

    results.append((T, ba, vrec))

best_T, best_ba, best_vrec = max(results, key=lambda x: x[1])
print(f"Best T_VAS (by val BA): {best_T:.3f} | val BA={best_ba:.4f} | val VascRecall={best_vrec:.4f}")

# 4) Retrained LR on FULL TRAIN and evaluate once on TEST using tuned threshold
clf_lr.fit(X_train, y_train)

class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

proba_test = clf_lr.predict_proba(X_test)
preds_test = []

for p in proba_test:
    if p[vas_idx] >= best_T:
        preds_test.append("Vascular")
    else:
        preds_test.append(class_order[np.argmax(p)])

print("\n=== Logistic Regression (Vascular threshold tuned) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels))
print("Balanced accuracy:", balanced_accuracy_score(y_test, preds_test))
print(confusion_matrix(y_test, preds_test, labels=labels))

Best T_VAS (by val BA): 0.074 | val BA=0.6574 | val VascRecall=0.6400

=== Logistic Regression (Vascular threshold tuned) ===
               precision    recall  f1-score   support

           AD     0.6767    0.5589    0.6122       603
NoDementia_Hy     0.9391    0.8629    0.8994      2217
       PD_LBD     0.5068    0.6379    0.5649        58
     Vascular     0.0700    0.7000    0.1273        30

     accuracy                         0.7937      2908
    macro avg     0.5482    0.6899    0.5509      2908
 weighted avg     0.8671    0.7937    0.8252      2908

Balanced accuracy: 0.6899202755915415
[[ 337  115   16  135]
 [ 146 1913   19  139]
 [  11    5   37    5]
 [   4    4    1   21]]


In [118]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# --------------------------------------------------
# 0) Labels / weights
# --------------------------------------------------
labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# Slightly reduced vascular weight to help precision/F1
custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 5.0,
    "Vascular": 6.0
}

# --------------------------------------------------
# 1) Train / validation split from TRAIN only
# --------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# --------------------------------------------------
# 2) Candidate model settings
#    One model change included: Elastic Net logistic regression
# --------------------------------------------------
model_settings = [
    {
        "name": "LR_L2_C0.1",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.1,
        "l1_ratio": None,
    },
    {
        "name": "LR_L2_C0.05",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.05,
        "l1_ratio": None,
    },
    {
        "name": "LR_ElasticNet_C0.1_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.1,
        "l1_ratio": 0.2,
    },
    {
        "name": "LR_ElasticNet_C0.05_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.05,
        "l1_ratio": 0.2,
    },
]

thresholds = np.linspace(0.05, 0.40, 36)

# --------------------------------------------------
# Helper: apply vascular threshold override
# --------------------------------------------------
def predict_with_vascular_threshold(proba, class_order, threshold):
    vas_idx = list(class_order).index("Vascular")
    preds = []
    for p in proba:
        if p[vas_idx] >= threshold:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    return np.array(preds)

# --------------------------------------------------
# 3) Validation search:
#    choose model + threshold by vascular F1
#    tie-breakers: macro F1, balanced accuracy
# --------------------------------------------------
search_rows = []

for cfg in model_settings:
    clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            max_iter=12000,
            class_weight=custom_w,
            C=cfg["C"],
            penalty=cfg["penalty"],
            solver=cfg["solver"],
            l1_ratio=cfg["l1_ratio"],
            n_jobs=None
        ))
    ])

    clf.fit(X_tr, y_tr)

    proba_val = clf.predict_proba(X_val)
    class_order = clf.named_steps["model"].classes_

    for T in thresholds:
        preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

        # overall metrics
        ba = balanced_accuracy_score(y_val, preds_val)
        macro_f1 = f1_score(y_val, preds_val, average="macro")

        # per-class metrics
        p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
            y_val, preds_val, labels=labels, zero_division=0
        )

        metric_map = {
            lab: {"precision": p, "recall": r, "f1": f, "support": s}
            for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
        }

        vprec = metric_map["Vascular"]["precision"]
        vrec  = metric_map["Vascular"]["recall"]
        vf1   = metric_map["Vascular"]["f1"]

        search_rows.append({
            "model_name": cfg["name"],
            "penalty": cfg["penalty"],
            "solver": cfg["solver"],
            "C": cfg["C"],
            "l1_ratio": cfg["l1_ratio"],
            "threshold": T,
            "val_bal_acc": ba,
            "val_macro_f1": macro_f1,
            "val_vasc_precision": vprec,
            "val_vasc_recall": vrec,
            "val_vasc_f1": vf1,
        })

search_df = pd.DataFrame(search_rows)

# Sort by vascular F1 first, then macro F1, then BA
search_df = search_df.sort_values(
    ["val_vasc_f1", "val_macro_f1", "val_bal_acc"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Top validation settings:")
display(search_df.head(10))

best_row = search_df.iloc[0]
best_model_name = best_row["model_name"]
best_T = float(best_row["threshold"])

print(
    f"Best setting: {best_model_name} | "
    f"T_VAS={best_T:.3f} | "
    f"val VascF1={best_row['val_vasc_f1']:.4f} | "
    f"val MacroF1={best_row['val_macro_f1']:.4f} | "
    f"val BA={best_row['val_bal_acc']:.4f} | "
    f"val VascPrec={best_row['val_vasc_precision']:.4f} | "
    f"val VascRec={best_row['val_vasc_recall']:.4f}"
)


c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packag

Top validation settings:


,model_name,penalty,solver,C,l1_ratio,threshold,val_bal_acc,val_macro_f1,val_vasc_precision,val_vasc_recall,val_vasc_f1
0,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.36,0.637566,0.613418,0.204545,0.36,0.260870
1,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.27,0.644832,0.611738,0.192308,0.40,0.259740
2,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.32,0.637047,0.612155,0.200000,0.36,0.257143
3,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.33,0.637047,0.612155,0.200000,0.36,0.257143
4,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.34,0.637047,0.612155,0.200000,0.36,0.257143
5,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.35,0.637047,0.612155,0.200000,0.36,0.257143
6,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.31,0.636906,0.611172,0.195652,0.36,0.253521
7,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.30,0.636906,0.610463,0.191489,0.36,0.250000
8,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.26,0.644172,0.608461,0.178571,0.40,0.246914
9,LR_ElasticNet_C0.05_l1r0.2,elasticnet,saga,0.05,0.2,0.38,0.628744,0.610305,0.200000,0.32,0.246154


Best setting: LR_ElasticNet_C0.05_l1r0.2 | T_VAS=0.360 | val VascF1=0.2609 | val MacroF1=0.6134 | val BA=0.6376 | val VascPrec=0.2045 | val VascRec=0.3600


In [198]:
# --------------------------------------------------
# 4) Refit best model on FULL TRAIN
# --------------------------------------------------

# clf_best = Pipeline(steps=[
#     ("preprocess", preprocess),
#     ("model", LogisticRegression(
#         max_iter=12000,
#         class_weight=custom_w,
#         C=0.1,
#         penalty="elasticnet",
#         solver="saga",
#         l1_ratio=0.2,
#         n_jobs=None
#     ))
# ])


# best_T = 0.330

clf_best = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=12000,
        class_weight=custom_w,
        C=0.05,
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        n_jobs=None
    ))
])


best_T = 0.360

clf_best.fit(X_train, y_train)

class_order = clf_best.named_steps["model"].classes_
proba_test = clf_best.predict_proba(X_test)
preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# --------------------------------------------------
# 5) Final test evaluation
# --------------------------------------------------
print("\n=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

test_bal_acc = balanced_accuracy_score(y_test, preds_test)
test_macro_f1 = f1_score(y_test, preds_test, average="macro")
test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")
test_cm = confusion_matrix(y_test, preds_test, labels=labels)

print("Balanced accuracy:", test_bal_acc)
print("Macro F1:", test_macro_f1)
print("Weighted F1:", test_weighted_f1)
print(confusion_matrix(y_test, preds_test, labels=labels))

# Optional: print vascular-specific test metrics cleanly
p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
    y_test, preds_test, labels=labels, zero_division=0
)
test_metric_map = {
    lab: {"precision": p, "recall": r, "f1": f, "support": s}
    for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
}

print("\nTest vascular metrics:")
print(test_metric_map["Vascular"])

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===
               precision    recall  f1-score   support

           AD     0.6610    0.7081    0.6837       603
NoDementia_Hy     0.9326    0.8926    0.9122      2217
       PD_LBD     0.4875    0.6724    0.5652        58
     Vascular     0.2000    0.4000    0.2667        30

     accuracy                         0.8449      2908
    macro avg     0.5703    0.6683    0.6070      2908
 weighted avg     0.8599    0.8449    0.8512      2908

Balanced accuracy: 0.6682968879336848
Macro F1: 0.6069557012050181
Weighted F1: 0.8512418006659954
[[ 427  131   20   25]
 [ 197 1979   20   21]
 [  12    5   39    2]
 [  10    7    1   12]]

Test vascular metrics:
{'precision': np.float64(0.2), 'recall': np.float64(0.4), 'f1': np.float64(0.26666666666666666), 'support': np.int64(30)}


c:\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [44]:
import joblib
import pandas as pd

# Save model bundle
joblib.dump(
    {
        "model": clf_best,
        "threshold": best_T,
        "class_order": list(class_order),
        "feature_cols": list(X_train.columns)
    },
    "../output/dementia_classifier_bundle.joblib"
)

print("Model bundle saved.")

# Predict on test set
proba = clf_best.predict_proba(X_test)
preds = predict_with_vascular_threshold(proba, class_order, best_T)

# Export predictions with true NACCID from df4
clf_preds = pd.DataFrame({
    "NACCID": df4.loc[X_test.index, "NACCID"].values,
    "predicted_class": preds
})

# Optional: save probabilities too
for i, cls in enumerate(class_order):
    clf_preds[f"prob_{cls}"] = proba[:, i]

clf_preds["vascular_threshold"] = best_T

clf_preds.to_csv("../output/classifier_predictions.csv", index=False)

print("Saved ../output/classifier_predictions.csv")
display(clf_preds.head())

Model bundle saved.
Saved ../output/classifier_predictions.csv


,NACCID,predicted_class,prob_AD,prob_NoDementia_Hy,prob_PD_LBD,prob_Vascular,vascular_threshold
0,NACC056862,NoDementia_Hy,0.172698,0.795219,0.008433,0.023651,0.4
1,NACC774813,NoDementia_Hy,0.123723,0.856753,0.013818,0.005706,0.4
2,NACC989206,NoDementia_Hy,0.130130,0.852813,0.003812,0.013245,0.4
3,NACC735141,AD,0.572949,0.145826,0.218836,0.062389,0.4
4,NACC252819,AD,0.664147,0.227580,0.029827,0.078446,0.4
